In [1]:
import numpy as np
from numba import njit, typed, types,jit
import xarray as xr
import timeit

In [2]:


# ---------- helper Numba types ----------
coord_type    = types.UniTuple(types.int16, 2)        # (row, col)
list_type     = types.ListType(coord_type)            # list of coordinates
array2d_type  = types.int16[:, ::1]                   # (2, n_pts) C-contiguous
dict_lists_t  = types.DictType(types.int64, list_type)
dict_arrays_t = types.DictType(types.int64, array2d_type)

@njit
def extract_cloud_coordinates(cloudtracknumber_field,   # 3-D, shape (1, ny, nx)
                              cloud_id_in_field,        # 1-D array of unique IDs
                              max_size):                # per-cloud hard cap
    """
    Returns a Dict[int -> int16[:, ::1]]
        key   : cloud ID
        value : 2×N array with the exact #pixels (N ≤ max_size)
                 row coords in axis=0, col coords in axis=1
    Memory use ≈ Σ( N_cloud × 2 × 2 bytes ) with zero over-allocation.
    """

    # -- first pass: collect coordinates in typed.Lists --------------------
    coord_lists = typed.Dict.empty(                     # type: Dict[int, List[(int16,int16)]]
        key_type   = types.int64,
        value_type = list_type
    )

    ny, nx = cloudtracknumber_field.shape[1:]

    for row in range(ny):
        for col in range(nx):
            cid = cloudtracknumber_field[0, row, col]
            if cid == 0:
                continue          # background pixel – ignore

            if cid not in coord_lists:
                coord_lists[cid] = typed.List.empty_list(coord_type)

            lst = coord_lists[cid]
            if len(lst) < max_size:              # honour the user-supplied cap
                lst.append((np.int16(row), np.int16(col)))

    # -- second pass: pack each list into a perfectly-sized 2×N array ------
    result = typed.Dict.empty(                     # type: Dict[int, int16[:,::1]]
        key_type   = types.int64,
        value_type = array2d_type
    )

    for cid in coord_lists:
        lst = coord_lists[cid]
        n   = len(lst)
        arr = np.empty((2, n), dtype=np.int16)

        for i in range(n):
            rc = lst[i]
            arr[0, i] = rc[0]     # row
            arr[1, i] = rc[1]     # col

        result[cid] = arr

    return result


In [12]:
@njit
def extract_cloud_coordinates_old(cloudtracknumber_field, cloud_id_in_field, max_size):
    # Define the dictionary with the appropriate types
    loc_hash_map_cloud_numbers = {
        j: (0, np.zeros((2, max_size), dtype=np.int16)) for j in cloud_id_in_field}
    # # Traverse the 3D array
    # for i in cloud_id_in_field:
    #     loc_hash_map_cloud_numbers[val] = (0,np.empty((2,max_size),dtype=np.int16))
    for row in range(cloudtracknumber_field.shape[1]):
        for col in range(cloudtracknumber_field.shape[2]):
            val = cloudtracknumber_field[0, row, col]
            if val != 0:
                ind, cord = loc_hash_map_cloud_numbers[val]
                if ind <= max_size:
                    cord[:, ind] = np.asarray([row, col], dtype=np.int16)
                    ind += 1
                    # print(ind)
                    loc_hash_map_cloud_numbers[val] = (ind, cord)
    return loc_hash_map_cloud_numbers
    # return loc_hash_map_cloud_numbers

In [13]:
test_arr = np.zeros((1, 5, 5), dtype=np.int16)
test_arr[0, 1:2, 1:3] = 1
test_arr[0, 3:5, 2:5] = 2
test_arr[0, 2:4, 1:3] = 3
test_arr

array([[[0, 0, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 3, 3, 0, 0],
        [0, 3, 3, 2, 2],
        [0, 0, 2, 2, 2]]], dtype=int16)

In [4]:
result_new = extract_cloud_coordinates(test_arr,np.array([1,2]), 10)

In [14]:
result_old = extract_cloud_coordinates_old(test_arr,np.array([1,2,3]), 10)

In [5]:
data = xr.open_dataset("/cluster/work/climate/dnikolo/Job_output/np/Agg_03_T_06_00/20070101.0000_20070115.0000/pixel_path_tracking/20070101.0000_20070115.0000/cloudtracks_20070106_134500.nc")

In [9]:
cloudtracknumber = data["tracknumber"].data
cloudtracknumber[np.isnan(cloudtracknumber)] = 0
cloudtracknumber=cloudtracknumber.astype(int)
cloud_id_in_field, counts = np.unique(
            cloudtracknumber, return_counts=True)
counts = counts[cloud_id_in_field != 0]
cloud_id_in_field = cloud_id_in_field[cloud_id_in_field != 0]

In [10]:
result_new_2 = extract_cloud_coordinates(cloudtracknumber,cloud_id_in_field, counts.max()+1)

In [15]:
result_old = extract_cloud_coordinates_old(cloudtracknumber,cloud_id_in_field, counts.max())

In [23]:
# compute the “counts.max()+1” just once, if you don’t want it re-computed on each timing cycle
max_count = counts.max() + 1

t_new = timeit.timeit(
    lambda: extract_cloud_coordinates(cloudtracknumber,
                                      cloud_id_in_field,
                                      max_count),
    number=100
)

In [24]:
# compute the “counts.max()+1” just once, if you don’t want it re-computed on each timing cycle
max_count = counts.max() + 1

t_old = timeit.timeit(
    lambda: extract_cloud_coordinates_old(cloudtracknumber,
                                      cloud_id_in_field,
                                      max_count),
    number=100
)

In [25]:
t_new-t_old

-0.10711096692830324

In [5]:
result[3]

array([[2, 2, 3, 3],
       [1, 2, 1, 2]], dtype=int16)